Disclaimer: None of this is real data, is was generated by Claude AI and is intended to simulate real world messy data. The data does not represent any real individuals and any resemblance is purely coincidence.

In [2]:
%cd /content
import os
if not os.path.exists("valley-outfitters-analysis"):
    !git clone https://github.com/LSRajan/valley-outfitters-analysis.git
%cd valley-outfitters-analysis

import pandas as pd
pd.set_option("display.max_columns", None)

/content
Cloning into 'valley-outfitters-analysis'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 60 (delta 16), reused 49 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 3.90 MiB | 14.32 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/valley-outfitters-analysis


In [3]:
orders = pd.read_csv("data/raw/orders.csv")
customers = pd.read_csv("data/raw/customers.csv")
order_items = pd.read_csv("data/raw/order_items.csv")
products = pd.read_csv("data/raw/products.csv")
returns = pd.read_csv("data/raw/returns.csv")
web_sessions = pd.read_csv("data/raw/web_sessions.csv")

# 1. Orders

In [4]:
orders.head()

,order_id,customer_id,order_date,ship_date,status,payment_method,channel,discount_code,discount_amount,shipping_cost,order_total
0,ORD-223969,C3054,2025-09-24,2025-09-26,delivered,klarna,Web,NaN,0.00,0.0,208.08
1,ORD-221807,C1857,2026-01-28,NaN,Cancelled,Gift Card,Marketplace,NaN,0.00,0.0,223.48
2,ORD-210560,C5105,2025-12-18,2025-12-24,Delivered,Card,App,NaN,0.00,0.0,109.14
3,ORD-209414,C4333,2024-07-01,2024-07-06,Delivered,Card,app,welcome10,30.17,0.0,271.56
4,ORD-219246,C5677,27.06.24,"July 01, 2024",Delivered,Apple Pay,Web,NaN,0.00,0.0,591.92


In [5]:
orders.isna().sum()

,0
order_id,0
customer_id,383
order_date,346
ship_date,3257
status,0
payment_method,0
channel,0
discount_code,19138
discount_amount,0
shipping_cost,0


### Issue 01 : Duplicate rows in orders table

In [6]:
orders.duplicated().sum()

np.int64(90)

### Proposed Fix 01: There are 90 duplicated rows in the table. Solve by dropping duplicated rows.

In [7]:
orders_temp = orders.drop_duplicates().copy()

## 1.1 order_id

In [8]:
orders["order_id"].head()

,order_id
0,ORD-223969
1,ORD-221807
2,ORD-210560
3,ORD-209414
4,ORD-219246


### Check: order_id should be unique with no duplicates

In [9]:
orders_temp["order_id"].duplicated().sum()

np.int64(0)

### Check: order_id should be in the format "ORD-[id number]"

In [10]:
pattern = r"ORD-\d+"
check = orders_temp["order_id"].str.fullmatch(pattern, na = False)
check.all()

np.True_

## 1.2 customer_id

### Issue 02 : customer_id contains missing values.

In [11]:
print(orders_temp["customer_id"].isna().sum())
print(str(round(orders["customer_id"].isna().mean()*100, 2)) + "% of customer ids are missing")

382
1.53% of customer ids are missing


### Proposed Fix 02: A missing customer_id row does not make the order itself invalid, but it does break the relationship between the orders table and customers table. Any calculations involving this relationship should take this discrepancy into account, otherwise the order information can still be used without a customer id, so these rows will not be removed.


### Issue 03 : customer_id has inconsistent formatting

In [12]:
pattern = r"C\d+"
check = orders_temp["customer_id"].str.fullmatch(pattern, na = True)
check.all()

np.False_

In [13]:
orders_temp.loc[~check, "customer_id"]

,customer_id
96,c1730
293,c5806
1141,c5622
1418,c5260
1951,c2060
...,...
24282,c1545
24347,c5689
24780,c3382
24943,c5117


### Proposed Fix 03: The customer ids that do not match the format just have a lowercase prefix. To solve, capitalize all customer IDs

In [14]:
orders_temp["customer_id"] = orders_temp["customer_id"].str.upper()

In [15]:
pattern = r"C\d+"
check = orders_temp["customer_id"].str.fullmatch(pattern, na = True)
check.all()
# There are no remaining formatting issues after capitalising.

np.True_

### Issue 04 : All customer ids in the orders table should exist inside of the customers table, but many do not.

In [16]:
customers["customer_id"].head()

,customer_id
0,C3082
1,C5810
2,C6231
3,C5373
4,C3666


In [17]:
pattern = r"C\d+"
check = customers["customer_id"].str.fullmatch(pattern, na = True)
check.all()
#all customer ids in the customers table fit the pattern, so can be compared directly

np.True_

In [18]:
check = orders_temp["customer_id"].isin(customers["customer_id"])
check.all()

np.False_

In [19]:
not_present = orders_temp.loc[~check & orders_temp["customer_id"].notna(), "customer_id"]
print(not_present.count())
print(not_present.nunique())

266
236


### Proposed Fix 04: There are 266 customer ids (236 unique) that are valid in format but do not match a customer id in the customers table. These ids can still be kept to keep a record and as it may be useful if more data was provided. However, these inconsistencies should be accounted for during exploration.

## 1.3 order_date

### Issue 05 : There are missing order dates (more are missing after replacing non-missing NA values with NA)

In [20]:
orders_temp["order_date"].isna().sum()

np.int64(344)

### Proposed Fix 05: Similar to before, individual missing values do not lose too much information and the row remains useable. These values/rows do not need adjustment.

### Issue 06 : Order date should have consistent formatting. Instead it has several values representing missing values, date-time instead of just date, and different representations of dates. Standard date should be YYYY-MM-DD.

In [21]:
orders_temp["order_date"].value_counts()

,count
order_date,
2026-09-30,103
0000-00-00,66
TBC,65
unknown,62
2026-09-14,44
...,...
2026-03-13 15:45:56,1
2026-08-21 11:20:12,1
2025-12-28 17:24:58,1


### (Partial) Proposed Fix 06: There are several non-missing NA values that can be replaced with NA. This would keep the values in order date as date types and keep consistent formatting for unknown order dates.

In [22]:
non_missing_na = ["TBC", "unknown", "0000-00-00"]
orders_temp["order_date"] = orders_temp["order_date"].replace(non_missing_na, pd.NA)

In [23]:
with pd.option_context("display.max_rows", None):
    display(orders_temp["order_date"].value_counts())

,count
order_date,
2026-09-30,103
2026-09-14,44
2026-07-29,35
2025-12-25,35
2026-09-21,35
2026-09-10,35
2026-07-31,33
2025-11-09,32
2025-11-12,32


In [24]:
pattern = r"\d{4}-\d{2}-\d{2}"
check = orders_temp["order_date"].str.fullmatch(pattern, na=False)
incorrect_dates = orders_temp.loc[~check & orders_temp["order_date"].notna(), "order_date"]
incorrect_dates.value_counts()

,count
order_date,
30 Sep 2026,14
30.09.26,14
"September 30, 2026",11
09-30-2026,10
30/09/2026,9
...,...
2024-09-25 17:20:35,1
2026-02-05 15:12:27,1
31.10.24,1


### Proposed Fix 06: The remaining values are just in the wrong time formats. Get the values by their format and convert those into the correct format. Any remaining values not in a given format can be handled manually.

## 1.4 ship_date

### Issue 07 : (Same issues as order date) Ship date should have consistent formatting. Instead it has several values representing missing values, date-time instead of just date, and different representations of dates. Standard date should be YYYY-MM-DD. Many dates are invalid too.

In [25]:
orders_temp["ship_date"].value_counts()

,count
ship_date,
TBC,52
0000-00-00,50
2025-11-09,49
2026-07-23,48
unknown,45
...,...
15.05.25,1
"March 31, 2024",1
"July 01, 2024",1


### Proposed Fix 07 : Same fix as previous date fixes. Replace invalid dates with NA.

## 1.5 status

### Issue 08 : While status has a fixed set of possible values, the rows contain many variations, mispellings, or capitalisations of the same value.

In [26]:
orders_temp["status"].value_counts()

,count
status,
Delivered,14899
Cancelled,1348
completed,1036
DELIVERED,1016
Complete,991
Returned,988
Deliverd,987
delivered,986
Shipped,775


In [27]:
orders_temp["status"].str.strip().str.lower().value_counts()

,count
status,
delivered,16901
cancelled,1590
returned,1206
completed,1036
complete,991
deliverd,987
shipped,926
pending,512
processing,391


### Proposed Fix 08: These values can be stripped and lowered, followed by manually fixing spelling errors and ensuring one consistent label rather than several labels for the same underlying meaning.

## 1.6 payment_method

### Issue 09 : While payment method has a fixed set of possible values, the rows contain many variations, mispellings, or capitalisations of the same value. There are also different versions for abbreviated or spaced values.

In [28]:
orders_temp["payment_method"].value_counts()

,count
payment_method,
PayPal,4041
Apple Pay,4035
Gift Card,4003
Klarna,3968
Card,3927
klarna,508
Giftcard,506
ApplePay,501
apple pay,492


### Proposed Fix 09: These values can be stripped and lowered, followed by manually fixing spelling errors and ensuring one consistent label rather than several labels for the same underlying meaning.

## 1.7 channel

### Issue 10 : While channel has a fixed set of possible values, the rows contain many variations, mispellings, or capitalisations of the same value. There are also different versions for abbreviated or spaced values.

In [29]:
orders_temp["channel"].value_counts()

,count
channel,
Web,11216
App,5529
Marketplace,2685
In-store,1148
web,811
WEB,804
Online,797
Phone,660
APP,379


### Proposed Fix 10: These values can be stripped and lowered, followed by manually fixing spelling errors and ensuring one consistent label rather than several labels for the same underlying meaning.

## 1.8 discount_code

### Issue 11 : The discount code column appears to have many different variations of the same discount code. For example, 'welcome10', 'Welcome10', 'WELCOME10'. The reasonable assumption is that discount codes are not case sensitive and that these are all the same code. However it should be noted that they might actually be distinct, and that might cause an error later on.

In [30]:
orders_temp["discount_code"].value_counts()

,count
discount_code,
WELCOME10,1709
SUMMER20,1301
FREESHIP,1024
LOYAL15,872
BF25,128
welcome10,110
WELCOME10,101
Welcome10,101
Summer20,76


### Proposed Fix 11: These values can be stripped and lowered, followed by manually fixing spelling errors and ensuring one consistent label rather than several labels for the same underlying discount code. Possible downstream errors should be considered if an issue arises.

## 1.9 discount_amount

### Check: discount amount should be float values to two decimal points

In [31]:
with pd.option_context("display.max_rows", None):
    display(orders_temp["discount_amount"].value_counts())

,count
discount_amount,
0.00,20278
9.76,12
1.96,11
4.55,11
4.54,10
0.72,10
18.56,10
8.30,9
20.65,9


In [32]:
orders_temp["discount_amount"].astype(float)

,discount_amount
0,0.00
1,0.00
2,0.00
3,30.17
4,0.00
...,...
25084,0.00
25085,0.00
25087,0.00
25088,0.00


In [33]:
check = (orders_temp["discount_amount"].round(2) != orders_temp["discount_amount"]) & orders_temp["discount_amount"].notna()
orders_temp.loc[check, "discount_amount"]

,discount_amount


### Check Finding: discount amount column is already a float and checks show that no values are over 2 decimal places. Manual checks show there are no other issues.

### Check: There are no missing discount amounts

In [34]:
orders_temp["discount_amount"].isna().sum()

np.int64(0)

## 1.10 shipping_cost

### Check and Finding: Shipping costs are one of four values, each of which are valid. There are no missing values either

In [35]:
orders_temp["shipping_cost"].value_counts(dropna=False)

,count
shipping_cost,
0.00,20871
3.99,2472
4.99,1258
6.99,399


## 1.11 order_total

In [36]:
orders_temp["order_total"].value_counts()

,count
order_total,
131.45,47
88.57,47
83.64,46
181.99,46
133.17,45
...,...
415.88,1
927.30,1
140.10,1


### Issue 12 : order_total should be float values to two decimal places. However, there are values using the prefix of "£".

In [37]:
#orders_temp["order_total"].astype(float) generates error due to "£" sign

### Proposed Fix 12 : Replace the "£" symbol with empty string directly

In [38]:
orders_temp["order_total"] = orders_temp["order_total"].str.replace("£", "").astype(float)

In [39]:
orders_temp["order_total"]

,order_total
0,208.08
1,223.48
2,109.14
3,271.56
4,591.92
...,...
25084,378.52
25085,357.98
25087,465.40
25088,391.11


### Issue 13 : order_total should be positive, even for returned orders.

In [40]:
check = orders_temp["order_total"].astype(float) < 0
orders_temp.loc[check, ["order_total", "status"]]

,order_total,status
126,-49.75,Returned
199,-200.73,delivered
211,-245.57,Processing
226,-358.70,Deliverd
523,-23.43,Delivered
...,...,...
24377,-83.64,Delivered
24429,-344.62,Delivered
24642,-119.84,Delivered
24702,-3.41,Delivered


### Proposed Fix 13 : The values appear to be correct but just have the wrong sign. To clean, take the absolute value of order_total.

In [41]:
check = (orders_temp["order_total"].astype(float).round(2) != orders_temp["order_total"].astype(float)) & orders_temp["order_total"].notna()
orders_temp.loc[check, "order_total"]

,order_total


### Check Finding: order_total column has been converted to a float and checks show that no values are over 2 decimal places. Manual checks show there are no other issues.

## 1.12 cross-column considerations

### Check and Finding: While individual column missing values were considered, here we check if there are rows with many NA values. The check finds no row with more than 3 NA values, and only 121 rows with 3 or more missing values. Hence there is no row with too little information to be relevant. No change suggested.

In [42]:
orders_temp[orders_temp.isna().sum(axis=1) > 2]

,order_id,customer_id,order_date,ship_date,status,payment_method,channel,discount_code,discount_amount,shipping_cost,order_total
109,ORD-201756,C5063,NaN,NaN,Delivered,Klarna,Online,NaN,0.0,0.0,184.48
213,ORD-203539,NaN,2026-02-09,NaN,Processing,Gift Card,Web,NaN,0.0,0.0,159.21
245,ORD-204918,C6096,10.05.26,NaN,Delivered,Klarna,WEB,NaN,0.0,0.0,NaN
1232,ORD-200171,C2863,NaN,NaN,Pending,gift card,App,NaN,0.0,0.0,1428.33
1324,ORD-213341,C5551,2025-05-25,NaN,Processing,PayPal,Web,NaN,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
23573,ORD-202685,NaN,2026-03-20 18:02:28,NaN,Complete,CARD,Web,NaN,0.0,0.0,99.55
23689,ORD-216775,C5815,NaN,NaN,Cancelled,Card,Web,NaN,0.0,0.0,276.64
23768,ORD-214465,NaN,30/10/2024,NaN,Processing,PayPal,Marketplace,NaN,0.0,0.0,630.73
24642,ORD-218666,NaN,2024-12-16,NaN,Delivered,Card,Web,NaN,0.0,0.0,-119.84


In [43]:
orders_temp[orders_temp.isna().sum(axis=1) > 3]

,order_id,customer_id,order_date,ship_date,status,payment_method,channel,discount_code,discount_amount,shipping_cost,order_total


# 2. Customers

In [44]:
customers.head()

,customer_id,first_name,last_name,email,phone,signup_date,date_of_birth,city,postcode,country,marketing_opt_in
0,C3082,Mia,Evans,mia.evans90@outlook.com,07720 937945,06/07/2025,1955/05/19,Wrexham,LL11 9SA,UK,no
1,C5810,Oliver,Robinson,oliver.robinson@outlook.com,07159 902747,01/01/2026,1968-12-19,Swansea,SA5 5SN,United Kingdom,No
2,C6231,LILY,Jones,lily.jones@hotmail.co.uk,07453 098924,2026-04-04,1973-02-03,Belfast,bt23ls,england,0
3,C5373,Leo,McDonald,leo.mcdonald@gmail.com,07929696950,2025-01-22,19/05/1991,Manchester,M5 4DE,UK,y
4,C3666,George,Thompson,george.thompson90@icloud.com,07625-269687,2025-12-02,1983-01-06,London,E5 9RS,United Kingdom,y


In [45]:
customers.duplicated().sum()

np.int64(0)

In [46]:
customers.isna().sum()

,0
customer_id,0
first_name,0
last_name,0
email,97
phone,373
signup_date,69
date_of_birth,113
city,90
postcode,147
country,110


## 2.1 customer_id

### Check and Finding: There should be no duplicate, missing, or incorrectly formatted customer ids. Check shows there are not.

In [47]:
customers["customer_id"].duplicated().sum()

np.int64(0)

In [48]:
pattern = r"C\d+"
check = customers["customer_id"].str.fullmatch(pattern, na = False)
check.all()

np.True_

## 2.2 first_name

### Issue 14 : Customer first name should be consistently capitalised.

In [49]:
customers["first_name"].value_counts()

,count
first_name,
Leo,150
Ella,145
Cerys,144
Dylan,141
Priya,140
...,...
zainab,1
RHYS,1
Noah,1


In [50]:
customers["first_name"].str.strip().str.lower().value_counts()

,count
first_name,
leo,160
ella,155
priya,154
dylan,153
cerys,149
chloe,146
bethan,146
aisha,145
george,142


### Proposed Fix 14 : Strip and Lower first name to maintain consistent capitalisation. Re-capitalise the first letter if needed.

## 2.3 last_name

### Check and Finding: There should be no missing, or incorrectly formatted last names. There are none, and checks confirm all last names are valid last names with no other errors.

In [51]:
customers["last_name"].value_counts()

,count
last_name,
Evans,197
Singh,196
Price,194
Lewis,194
Ali,193
Taylor,191
Smith,190
O'Brien,185
Murphy,184


## 2.4 email

### Issue 15 : Customer emails should be unique, there are several repeat emails. Emails appear to be in the format of [first name].[lastname] which has led to different people holding the same email.

In [52]:
customers["email"].value_counts()

,count
email,
oliver.ali@icloud.com,4
seren.evans@btinternet.com,4
bethan.mcdonald@hotmail.co.uk,3
priya.taylor@yahoo.co.uk,3
owen.murphy@gmail.com,3
...,...
priya.hughes78@outlook.com,1
amelia.singh74@yahoo.co.uk,1
grace.thompson@hotmail.co.uk,1


In [53]:
customers.loc[customers["email"].duplicated() & customers["email"].notna()].sort_values("email")

,customer_id,first_name,last_name,email,phone,signup_date,date_of_birth,city,postcode,country,marketing_opt_in
1622,C2778,Aisha,Griffiths,aisha.griffiths@yahoo.co.uk,07363 496921,2026/04/04,1961-04-16,Wrexham,LL3 7HP,United Kingdom,No
2791,C5701,Aisha,Griffiths,aisha.griffiths@yahoo.co.uk,NaN,2023-08-18,1979-03-21,Exeter,EX10 9HH,United Kingdom,y
3081,C5493,Aisha,Johnson,aisha.johnson@outlook.com,+44 7140 165674,2026-07-20,1973/04/07,Manchester,M6 9JF,United Kingdom,1
3296,C6146,aisha,Jones,aisha.jones12@hotmail.co.uk,0773760,NaN,1993-08-15,Swansea,SA11 1YP,Wales,0
4503,C2455,Aisha,Jones,aisha.jones41@btinternet.com,0789064,2026-09-03,"August 31, 1981",London,NaN,Great Britain,No
...,...,...,...,...,...,...,...,...,...,...,...
3797,C6276,Tomasz,Walker,tomasz.walker@hotmail.co.uk,07555 453771,14.10.25,1955-01-18,Swansea,SA16 1LA,United Kingdom,Y
4661,C5384,Zainab,Green,zainab.green@outlook.com,07601 858749,2025-07-23,1991-11-11,Birmingham,B9 1JJ,United Kingdom,NaN
4127,C6211,zainab,Griffiths,zainab.griffiths@yahoo.co.uk,07619 522302,2024-01-12,22.11.59,Cardiff,CF13 9BS,United Kingdom,no
2822,C3212,Zainab,Hall,zainab.hall92@hotmail.co.uk,07325 409455,2024-02-10,1973-02-28,Bangor,LL6 5TY,United Kingdom,no


In [54]:
customers[customers[["email", "date_of_birth"]].duplicated(keep=False)].sort_values("email")

,customer_id,first_name,last_name,email,phone,signup_date,date_of_birth,city,postcode,country,marketing_opt_in
2417,C3507,Aisha,Jones,aisha.jones12@hotmail.co.uk,0773760,2025-11-12,1993-08-15,Swansea,SA8 4ZX,UK,FALSE
3296,C6146,aisha,Jones,aisha.jones12@hotmail.co.uk,0773760,NaN,1993-08-15,Swansea,SA11 1YP,Wales,0
1205,C4599,Amelia,Lewis,amelia.lewis10@btinternet.com,07585130748,2025-02-11,1973-01-10,NaN,L10 6HX,United Kingdom,1
5280,C6315,AMELIA,Lewis,amelia.lewis10@btinternet.com,07585130748,2025-03-17,1973-01-10,Liverpool,L4 3AB,United Kingdom,no
687,C5721,Amelia,Nowak,amelia.nowak96@yahoo.co.uk,NaN,2025/04/17,1960-01-28,Bristol,BS19 6XN,NaN,No
...,...,...,...,...,...,...,...,...,...,...,...
1877,C1607,Tomasz,Walker,tomasz.walker@hotmail.co.uk,07555 453771,2025-09-04,1955-01-18,Swansea,SA2 2WR,UK,TRUE
3797,C6276,Tomasz,Walker,tomasz.walker@hotmail.co.uk,07555 453771,14.10.25,1955-01-18,Swansea,SA16 1LA,United Kingdom,Y
717,C3791,Tomasz,Khan,NaN,07840 523404,2024-03-11,NaN,Bangor,LL14 4SH,United Kingdom,0
1736,C4171,Tomasz,O'Brien,NaN,07916824765,2024-03-14,NaN,Liverpool,L1 1GJ,United Kingdom,no


### Proposed Fix 15 : Duplicate emails dont necessarily mean the same customer. By using their date of birth we can find customers who have signed up twice. For these rows, we can keep only the row with the most recent sign up date and relabel or merge instances with the old customer id with the new one. The row with the most recent sign up date will have the most up to date information about the customer.

###Beyond this, the email is not relevant for any exploration and no further cleaning is needed.

## 2.5 phone

### Phone number is also not relevant for any exploration so can be dropped and any such errors are dropped too.

## 2.6 signup_date

### Issue 16: The signup date column has the same issues as other date columns. There are several non-NA values to represent unknowns and the formatting is inconsistent.

In [55]:
customers["signup_date"].value_counts()

,count
signup_date,
0000-00-00,12
TBC,12
unknown,12
2026-06-15,10
2024-01-30,9
...,...
2024/12/23,1
04.10.23,1
22/09/2026,1


### Proposed Fix 16:  There are several non-missing NA values that can be replaced with NA. This would keep the values in sign up date as date types and keep consistent formatting for unknown order dates.
###The remaining values are just in the wrong time formats. Get the values by their format and convert those into the correct format. Any remaining values not in a given format can be handled manually.



## 2.7 date_of_birth

### Issue 17: The date of birth column again has the same errors as in other date columns

In [56]:
customers["date_of_birth"].value_counts()

,count
date_of_birth,
1900-01-01,51
2031-04-12,51
0000-00-00,17
unknown,15
TBC,12
...,...
1958-12-01,1
1978-06-29,1
1964-06-17,1


### Proposed Fix 17:  Same fix as other date columns

## 2.8 city

### Issue 18: The city values have inconsistent capitalisation and spelling.

In [57]:
customers["city"].value_counts(dropna=False)

,count
city,
Bristol,288
Aberystwyth,284
Sheffield,281
Newport,276
Bath,273
Glasgow,273
Exeter,266
Swansea,264
Bangor,255


### Proposed Fix 18: Values can be stripped and lowered. Spelling mistakes can be fixed by manual assignments.

## 2.9 post_code

### Location data can be obtained from the city column. Post codes are otherwise not useful for further exploration and can be removed.

In [58]:
customers["postcode"].value_counts(dropna=False)

,count
postcode,
NaN,147
EH20 1PG,2
BT4 9RH,2
G14 3TU,2
SA18 9YG,2
...,...
NP11 1TT,1
G10 3RD,1
LL10 7YQ,1


## 2.10 country

### Issue 19: The country values are very limited and often incorrect

In [59]:
customers["country"].value_counts()

,count
country,
United Kingdom,3174
UK,948
GB,318
U.K.,245
Great Britain,196
Wales,186
england,143


In [60]:
customers[["country", "city"]].value_counts()

country         city       
United Kingdom  Bristol        171
                Aberystwyth    168
                Nottingham     166
                Sheffield      165
                Swansea        165
                              ... 
Wales           london           1
england         Bristol          1
Wales           manchester       1
                swansea          1
england         Edinburg         1
Name: count, Length: 193, dtype: int64

### Proposed Fix 19: The country values can be replaced by mapping the city values to a country within the UK.

## 2.11 marketing_opt_in

### Issue 20: The marketing opt in values are inconsistent despite having the same meaning.

In [61]:
customers["marketing_opt_in"].value_counts(dropna=False)

,count
marketing_opt_in,
N,470
No,464
0,460
no,458
Y,456
FALSE,437
1,436
Yes,435
y,434


### Proposed Fix 20: Relabel all values to their respective boolean True/False values. NA values stay NA.

# 3. Order Items

In [62]:
order_items.head()

,order_item_id,order_id,product_id,quantity,unit_price
0,521656,ORD-210489,P236,1,35.62
1,502093,ORD-201014,P196,1,132.05
2,538709,ORD-218719,P109,1,106.65
3,549040,ORD-223642,P184,2,275.89
4,533522,ORD-216198,P257,1,65.43


### Issue 21: The order items table has 250 duplicated rows.

In [63]:
order_items.duplicated().sum()

np.int64(250)

### Proposed Fix 21: Drop the 250 duplicated rows.

In [64]:
oi_temp = order_items.drop_duplicates().copy()

In [65]:
oi_temp.isna().sum()

,0
order_item_id,0
order_id,0
product_id,0
quantity,0
unit_price,539


## 3.1 order_item_id

### Check and Finding: The order_item id should be unique, not missing, and a single integer. Checks show no issues.

In [66]:
oi_temp["order_item_id"].duplicated().sum()

np.int64(0)

In [67]:
oi_temp["order_item_id"].value_counts()

,count
order_item_id,
502175,1
521656,1
502093,1
538709,1
549040,1
...,...
533147,1
520342,1
507869,1


In [68]:
oi_temp["order_item_id"].astype(int)

,order_item_id
0,521656
1,502093
2,538709
3,549040
4,533522
...,...
52148,522308
52149,542910
52150,531663
52151,543028


## 3.2 order_id

### Check and Finding: The orderid should be in the format "ORD-[id number]" and not missing. It should also have a corresponding value in the orders table. Checks show no issues.

In [69]:
pattern = r"ORD-\d+"
check = oi_temp["order_id"].str.fullmatch(pattern, na = False)
check.all()

np.True_

In [70]:
oi_temp["order_id"].isin(orders["order_id"]).all()

np.True_

## 3.3 product_id

### Issue 22: Several product ids are incorrectly formatted with a lowercase P.

In [71]:
pattern = r"P\d+"
check = oi_temp["product_id"].str.fullmatch(pattern, na = False)
check.all()

np.False_

In [72]:
oi_temp[~check]

,order_item_id,order_id,product_id,quantity,unit_price
157,501237,ORD-200588,p198,2,155.46
410,542820,ORD-220653,p112,1,133.17
686,538338,ORD-218539,p242,1,46.12
735,507255,ORD-203513,p203,1,121.91
829,537030,ORD-217920,p122,1,245.63
...,...,...,...,...,...
51506,519650,ORD-209500,p150,1,49.69
51534,540060,ORD-219346,p152,1,104.27
51572,512341,ORD-206014,p201,1,121.20
51877,500307,ORD-200148,p257,4,65.43


In [73]:
oi_temp["product_id"] = oi_temp["product_id"].str.upper()
check = oi_temp["product_id"].str.fullmatch(pattern, na = False)
check.all()

np.True_

### Proposed Fix 22: Capitalise the product id string

### Issue 23: The product id should exist within the products table, but 111 rows do not match this condition.

In [74]:
oi_temp["product_id"].isin(products["product_id"]).all()

np.False_

In [75]:
oi_temp[~oi_temp["product_id"].isin(products["product_id"])]

,order_item_id,order_id,product_id,quantity,unit_price
503,515633,ORD-207587,P970,1,42.60
1036,507366,ORD-203563,P913,1,45.38
1074,528668,ORD-213877,P976,1,145.19
1198,542865,ORD-220675,P945,1,239.47
1287,533514,ORD-216194,P973,4,225.73
...,...,...,...,...,...
49254,517565,ORD-208505,P970,1,59.18
49259,501095,ORD-200519,P955,1,97.59
50262,522546,ORD-210921,P958,2,48.60
50830,525675,ORD-212419,P925,1,184.48


### Proposed Fix 23: The product id should exist within the products table, but it doesnt necessarily impact the order items table. Removing these rows would harm the order totals, and keeping them does not neccesarily harm exploration as long as they taken into account. So these values can be kept.

## 3.4 quantity

### Issue 24: Quantity has negative values and string representations of numbers

In [76]:
oi_temp["quantity"].value_counts()

,count
quantity,
1,38708
2,8719
3,2499
4,993
5,505
0,163
-1,107
one,80
250,31


### Proposed Fix 24: Map string values to integers and take absolute values of quantities

In [77]:
oi_temp["quantity"] = oi_temp["quantity"].replace({"one":1, "two":2, "three":3})

In [78]:
oi_temp.loc[oi_temp["quantity"].astype(int) < 0]

,order_item_id,order_id,product_id,quantity,unit_price
1372,544329,ORD-221371,P124,-1,106.06
1708,514717,ORD-207158,P161,-1,246.25
2043,534672,ORD-216772,P158,-1,191.50
2085,505724,ORD-202794,P159,-2,310.13
2287,501382,ORD-200666,P175,-1,48.79
...,...,...,...,...,...
51415,506691,ORD-203252,P231,-1,34.69
51755,549208,ORD-223717,P225,-1,66.90
51805,544717,ORD-221560,P218,-1,31.11
52066,529946,ORD-214477,P134,-1,188.89


In [79]:
oi_temp["quantity"] = oi_temp["quantity"].astype(int).abs()

In [80]:
oi_temp["quantity"].value_counts()

,count
quantity,
1,38895
2,8757
3,2515
4,996
5,507
0,163
250,31
100,24
999,15


### Issue 25: Several quantities are significantly higher than reasonable. Some values are 0 which would not make sense for an order.



In [81]:
oi_temp.loc[oi_temp["quantity"] > 5]

,order_item_id,order_id,product_id,quantity,unit_price
824,546183,ORD-222295,P209,250,63.17
1917,508802,ORD-204260,P134,250,188.89
2653,549025,ORD-223634,P190,100,42.60
2813,530505,ORD-214745,P260,999,11.10
5078,525098,ORD-212145,P146,999,112.19
...,...,...,...,...,...
46193,523875,ORD-211566,P152,100,104.27
47669,535360,ORD-217110,P186,999,301.73
47986,538555,ORD-218643,P163,250,133.05
50661,546169,ORD-222284,P178,250,NaN


###Proposed Fix 25: Replace with NA


## 3.5 unit_price

### Issue 26: Unit price values have "£" symbol

In [82]:
oi_temp["unit_price"].value_counts()

,count
unit_price,
202.06,356
153.16,353
83.64,352
184.48,352
97.59,347
...,...
£190.29,1
£142.34,1
£225.73,1


### Proposed Fix 26: Replace with empty string

In [83]:
oi_temp["unit_price"] = oi_temp["unit_price"].str.replace("£", "").astype(float)

### Check and Finding: Unit prices are otherwise correctly formatted and float values.

In [84]:
check = (oi_temp["unit_price"].round(2) == oi_temp["unit_price"])
oi_temp.loc[~check & oi_temp["unit_price"].notna(), "unit_price"]

,unit_price


# 4. Products

In [85]:
products.head()

,product_id,product_name,category,unit_price,cost_price,stock_qty,supplier,launch_date,is_active
0,P101,Forest Waterproof Jacket,outerwear,220.20,131.29,369,Alpine Supply Co,"March 14, 2025",N
1,P102,Storm Waterproof Jacket,Outerwear,190.29,111.93,286,Northern Gear,2022-07-12,FALSE
2,P103,Ridge Waterproof Jacket,Outerwear,77.19,46.15,243,BlueRidge Textiles,2023-04-11,0
3,P104,Peak Waterproof Jacket,Outer wear,75.01,39.89,268,TrekCo Ltd,2022-10-19,N
4,P105,Coastal Insulated Parka,Outerwear,77.66,33.31,332,Northern Gear,2025-02-16,N


In [86]:
products.duplicated().sum()

np.int64(1)

In [87]:
products_temp = products.drop_duplicates().copy()

In [88]:
products_temp.isna().sum()

,0
product_id,0
product_name,0
category,0
unit_price,0
cost_price,0
stock_qty,0
supplier,0
launch_date,2
is_active,0


## 4.1 product_id

### Check and Finding: products are correctly formatted, unique and not missing

In [89]:
products_temp["product_id"].duplicated().sum()

np.int64(0)

In [90]:
pattern = r"P\d+"
check = products_temp["product_id"].str.fullmatch(pattern, na = True)
check.all()

np.True_

## 4.2 product_name

### Check and Finding: product names return no issues, but should be stripped and lowered for consistency

In [91]:
with pd.option_context("display.max_rows", None):
    display(products_temp["product_name"].str.strip().str.lower().value_counts())


,count
product_name,
peak windbreaker,3
alpine trail runner,3
ridge sleeping bag,3
valley softshell jacket,2
coastal insulated parka,2
ridge head torch,2
coastal trekking poles,2
coastal beanie,2
alpine insulated bottle,2


## 4.3 category

### Issue 27: Category labels have inconsistent formatting, spacing and capitalisation.

In [92]:
products_temp["category"].value_counts()

,count
category,
Camping,29
Outerwear,25
Accessories,24
Base Layers,22
Footwear,19
Backpacks,15
Foot wear,5
accessories,3
Accesories,3


### Proposed Fix 27: Strip, lower and relabel variations into the same category labels.

## 4.4 unit_price

### Issue 28: Some unit prices have "£" symbols and "," instead of "."

In [93]:
#products_temp["unit_price"].astype(float)
#returns error when converting 'could not convert string to float: '£172.97''

### Proposed Fix 28: replace symbols with their respective correct versions

In [94]:
products_temp["unit_price"] = products_temp["unit_price"].str.replace("£", "")

In [95]:
products_temp["unit_price"] = products_temp["unit_price"].str.replace(",", ".").astype(float)

### Check: unit prices are otherwise valid

In [96]:
check = (products_temp["unit_price"].round(2) != products_temp["unit_price"]) | (products_temp["unit_price"].astype(float) < 0)
products_temp.loc[check]

,product_id,product_name,category,unit_price,cost_price,stock_qty,supplier,launch_date,is_active


## 4.5 cost_price

### Check and Finding: cost price can be converted to float and is formatted correctly

In [97]:
products_temp["cost_price"].astype(float)

,cost_price
0,131.29
1,111.93
2,46.15
3,39.89
4,33.31
...,...
158,4.64
159,4.70
160,14.63
161,28.05


In [98]:
check = (products_temp["cost_price"].round(2) != products_temp["cost_price"]) | (products_temp["cost_price"].astype(float) < 0)
products_temp.loc[check]

,product_id,product_name,category,unit_price,cost_price,stock_qty,supplier,launch_date,is_active


## 4.6 stock_qty

In [99]:
products_temp["stock_qty"].value_counts()

,count
stock_qty,
268,3
384,3
317,3
146,3
62,2
...,...
24,1
283,1
309,1


### Issue 29: Some stock quantities are negative

In [100]:
check = (products_temp["stock_qty"] < 0)
products_temp.loc[check]

,product_id,product_name,category,unit_price,cost_price,stock_qty,supplier,launch_date,is_active
15,P116,Valley Fleece Jacket,Outerwear,181.99,109.06,-23,Alpine Supply Co,2023/12/10,Y
17,P118,Ridge Fleece Jacket,Outerwear,185.02,111.59,-30,TrekCo Ltd,2025/04/09,Y
25,P126,Summit Down Gilet,Outerwear,225.73,88.50,-5,Summit Wholesale,2023-03-04,FALSE
35,P136,Peak Walking Boot,Footwear,88.57,57.43,-6,Alpine Supply Company,2025-06-19,TRUE
47,P148,Ridge Winter Boot,Foot wear,97.70,53.60,-22,TREKCO LTD,2022-08-27,TRUE
61,P162,Coastal Sleeping Bag,Camping,312.22,111.31,-16,Alpine Supply Co,"January 07, 2023",TRUE
94,P195,Valley 45L Trekking Pack,Backpacks,97.45,44.91,-24,Alpine Supply Co,2025-07-30,Y
107,P208,Highland Merino Base Top,Base Layers,45.76,18.95,-20,TrekCo Ltd,11.08.24,0
147,P248,Alpine Insulated Bottle,Accessories,39.83,15.61,-1,Alpine Supply Co,2023-05-23,Y


### Proposed Fix 29: The most likely situation is that these values are incorrectly negative and should be positive. Stock quantity has no bearing on anything else within the database. Absolute value should be taken

## 4.7 supplier

### Issue 30: Supplier names appear in many variations

In [101]:
products["supplier"].value_counts()

,count
supplier,
Alpine Supply Co,30
Northern Gear,30
TrekCo Ltd,27
BlueRidge Textiles,25
Summit Wholesale,23
Alpine supply co.,6
northern gear,4
Summit Wholesale Ltd,3
Northern Gear Ltd,3


### Proposed Fix 30: strip and lower values, and relabel variations to have one consistent supplier name

## 4.8 launch_date

### Issue 31: Launch data column has same date issues as other date columns

In [102]:
products["launch_date"].value_counts()

,count
launch_date,
2023-04-11,2
2025-08-03,2
2023-04-17,2
2022-08-15,2
2024-02-02,2
...,...
2024-09-03,1
2022-06-22,1
2025-01-03,1


### Proposed Fix 31: Clean column using the same methods

## 4.9 is_active

### Issue 32: is active has inconsistent labelling

In [103]:
products["is_active"].value_counts()

,count
is_active,
N,40
TRUE,31
0,25
FALSE,24
1,22
Y,22


### Proposed Fix 32: Relabel all values to boolean True/False values

# 5 returns

In [104]:
returns.head()

,return_id,order_item_id,order_id,return_date,reason,refund_amount,resolution
0,R7001,548180,ORD-223229,2025-10-08,Too Small,155.46,REFUND
1,R7002,502171,ORD-201051,2024-02-19,not waterproof,96.48,Refund
2,R7003,539059,ORD-218869,2025-07-29,Faulty,213.87,Refund
3,R7004,528860,ORD-213965,TBC,Too Small,586.67,refund
4,R7005,511660,ORD-205685,2026-09-30,arrived damaged,20.76,refund


In [105]:
returns.duplicated().sum()

np.int64(0)

In [106]:
returns.isna().sum()

,0
return_id,0
order_item_id,0
order_id,0
return_date,38
reason,277
refund_amount,55
resolution,0


## 5.1 return_id

In [107]:
returns["return_id"].head()

,return_id
0,R7001
1,R7002
2,R7003
3,R7004
4,R7005


### Check and Finding: Return id isnt duplicated, missing, or incorrectly formatted. No issues.

In [108]:
pattern = r"R\d+"
check = returns["return_id"].str.fullmatch(pattern, na = False)
check.all()

np.True_

## 5.2 order_item_id

### Check and Finding: order_item_id is correctly formatted, validm and all ids have a respective id in the order items table

In [109]:
returns["order_item_id"].head()

,order_item_id
0,548180
1,502171
2,539059
3,528860
4,511660


In [110]:
returns["order_item_id"].astype(int) # runs with no errors, all ids are valid.

,order_item_id
0,548180
1,502171
2,539059
3,528860
4,511660
...,...
2794,507671
2795,527210
2796,550817
2797,536628


In [111]:
check = returns["order_item_id"].isin(order_items["order_item_id"])
returns.loc[~check]

,return_id,order_item_id,order_id,return_date,reason,refund_amount,resolution


## 5.3 order_id

### Check and Finding: All order ids are in the correct format and have a respective value in the orders table

In [112]:
pattern = r"ORD-\d+"
check = returns["order_id"].str.fullmatch(pattern, na = False)
check.all()

np.True_

In [113]:
check = returns["order_id"].isin(order_items["order_id"])
returns.loc[~check]

,return_id,order_item_id,order_id,return_date,reason,refund_amount,resolution


## 5.4 return_date

### Issue 33: Return date has the same formatting and non-NA missing values issues as previous date columns

In [114]:
returns["return_date"].value_counts()

,count
return_date,
2026-09-30,87
2025-12-09,11
"September 30, 2026",11
unknown,10
2026-07-14,10
...,...
2025/09/05,1
2025-03-06,1
12/01/2025,1


### Proposed Fix 33: Fix date column in the same way as others.

## 5.5 reason

### Issue 34: Return reasons have different formatting and styles. There are also different reasons that relate to the same problem. As well as this, there are 277 NaN values.

In [115]:
returns["reason"].value_counts(dropna=False)

,count
reason,
NaN,277
didn't fit,140
wrong size,131
TOO SMALL,124
didnt fit,124
Doesn't fit,120
found cheaper,119
too big,117
Too Small,112


### Proposed Fix 34: Strip and lower values, replace values with more consistent category labelling. Relabel non-Na missing values too.

## 5.6 refund_amount

In [116]:
returns["refund_amount"].astype(float)

,refund_amount
0,155.46
1,96.48
2,213.87
3,586.67
4,20.76
...,...
2794,36.54
2795,45.38
2796,45.76
2797,103.25


### Check and Finding: No refunds amount have incorrect formatting.

In [117]:
check = ((returns["refund_amount"].round(2) != returns["refund_amount"]) | (returns["refund_amount"].astype(float) < 0)) & returns["refund_amount"].notna()
returns.loc[check]

,return_id,order_item_id,order_id,return_date,reason,refund_amount,resolution


## 5.7 Resolution

### Issue 35: resolution labels have variations of the same label

In [118]:
returns["resolution"].value_counts()

,count
resolution,
refund,580
REFUND,571
Store credit,568
Refund,543
Exchange,537


In [119]:
returns["resolution"].str.lower().str.strip().value_counts()

,count
resolution,
refund,1694
store credit,568
exchange,537


### Proposed Fix 35: strip and lower values

# 6 web_sessions

In [120]:
web_sessions.head()

,session_id,customer_id,session_timestamp,device,traffic_source,pages_viewed,session_duration_sec,converted
0,S909710,C3468,2024-11-05 20:21:08,Desktop,paid_search,1,62.0,0
1,S947222,NaN,2026-08-31 12:18:08,Mobile,Email,13,624.0,0
2,S926195,NaN,2025-06-17T12:02:46Z,Desktop,Google Ads,1,30.0,0
3,S901027,C4173,2026-07-23 13:02:00,mobile,social,1,42.0,0
4,S918483,NaN,2026-03-15 18:07:14,Mobile,paid_search,3,119.0,0


In [121]:
web_sessions.duplicated().sum()

np.int64(400)

In [122]:
web_temp = web_sessions.drop_duplicates().copy()

In [123]:
web_temp.isna().sum()

,0
session_id,0
customer_id,35736
session_timestamp,0
device,0
traffic_source,0
pages_viewed,0
session_duration_sec,2426
converted,0


## 6.1 session_id

### Check and Finding: session id has no missing, duplicated or incorrectly formatted values. No issues.

In [124]:
web_temp["session_id"]

,session_id
0,S909710
1,S947222
2,S926195
3,S901027
4,S918483
...,...
60395,S959989
60396,S907026
60397,S930182
60398,S913030


In [125]:
web_temp["session_id"].duplicated().sum()

np.int64(0)

In [126]:
pattern = r"S\d+"
check = web_temp["session_id"].str.fullmatch(pattern, na = False)
check.all()

np.True_

## 6.2 customer_id

In [127]:
web_temp["customer_id"]

,customer_id
0,C3468
1,NaN
2,NaN
3,C4173
4,NaN
...,...
60395,C2023
60396,NaN
60397,NaN
60398,NaN


### Issue 36: Nearly 60% of rows have a missing customer id

In [128]:
web_temp["customer_id"].isna().mean()

np.float64(0.5956)

### Proposed Fix 36: The customer_id column can be left as is. It represents the customer id of a user who has logged onto that session, but the missing values represent that around 60% of users do not use an account.

### Check and Finding : Customer ids are correctly formatted and all exist within the customers table.

In [129]:
not_na_ids = web_temp.loc[web_temp["customer_id"].notna(), "customer_id"]
pattern = r"C\d+"
check = not_na_ids.str.fullmatch(pattern, na = False)
check.all()

np.True_

In [130]:
check = web_temp["customer_id"].isin(customers["customer_id"])
web_temp[~check & web_temp["customer_id"].notna()]

,session_id,customer_id,session_timestamp,device,traffic_source,pages_viewed,session_duration_sec,converted


## 6.3 session_timestamp

### Issue 37: Session timestamp has several different formats for date and time

In [131]:
web_temp["session_timestamp"].value_counts()

,count
session_timestamp,
13/11/2025 04:13,2
20/04/2025 05:43,2
30/05/2025 17:59,2
07/09/2025 22:31,2
10/07/2025 07:50,2
...,...
2026-07-30 09:03:13,1
2025-08-12 15:15:41,1
2025-05-13 07:58:50,1


### Proposed Fix 37: Re-format each date using the same method as other date columns and create a seperate date column. Re-format timestamp, keeping it consistent in hour, minute (and seconds if needed) format and create a seperate timestamp column.

## 6.4 device

### Issue 38: device has many different variations of the same value. Inconsistent capitalisation and labels. Some labels might be better off under other labels.

In [132]:
web_temp["device"].value_counts()

,count
device,
Mobile,30534
Desktop,16825
Tablet,3569
MOBILE,1841
iPhone,1823
mobile,1795
desktop,1031
DESKTOP,997
PC,950


In [133]:
web_temp["device"].str.strip().str.lower().value_counts()

,count
device,
mobile,34170
desktop,18853
tablet,3986
iphone,1823
pc,950
ipad,218


### Proposed Fix 38: Strip and lower values. Remap labels that could be part of other labels.

## 6.5 traffic_source

### Issue 39: Traffic source has inconsistent formatting, capitalisation and several variations of the same source.

In [134]:
web_temp["traffic_source"].value_counts()

,count
traffic_source,
organic,10371
paid_search,6527
social,4193
direct,3619
Organic,3608
email,3567
organic search,3557
Google Organic,3462
google_ads,1700


### Proposed fix 39: Strip and lower values. Remap labels that could be part of other labels and to have consistent formatting. Relabel non-missing NA values.

## 6.6 pages_viewed

In [135]:
web_temp["pages_viewed"].value_counts()

,count
pages_viewed,
1,22726
2,7684
3,6059
4,4745
5,3835
...,...
854,1
469,1
587,1


### Issue 40: some rows have negative pages viewed, which is impossible. Others are significantly higher than is likely to be reasonable

In [136]:
check = web_temp["pages_viewed"].notna() & (web_temp["pages_viewed"].astype(int) > 0)
web_temp[~check]

,session_id,customer_id,session_timestamp,device,traffic_source,pages_viewed,session_duration_sec,converted
1086,S905424,NaN,2024-05-11 01:31:21,Mobile,E-mail,-5,282.0,0
1673,S901663,NaN,2026-04-25 23:48:38,Mobile,newsletter,-2,31.0,0
2637,S949024,NaN,2026-05-14 20:29:57,Mobile,Referral,-6,179.0,FALSE
2828,S912735,C2770,2025-04-02 22:20:05,Mobile,email,-2,99.0,0
3746,S949325,NaN,14/05/2025 10:45,DESKTOP,paid_search,-8,320.0,0
...,...,...,...,...,...,...,...,...
58750,S944203,NaN,15/06/2026 16:38,Mobile,organic,-4,251.0,1
58794,S918529,C2746,2025-11-15 09:51:06,desktop,organic,-1,68.0,0
59300,S925005,NaN,2025-07-31 17:20:46,Desktop,paid_search,-5,128.0,0
59544,S937097,NaN,2025-09-09T02:06:58Z,Desktop,Google Ads,-1,64.0,0


### Proposed Fix 40: Take the absolute value of these values.

In [137]:
web_temp["pages_viewed"] = web_temp["pages_viewed"].abs()

In [138]:
check = web_temp["pages_viewed"].notna() & (web_temp["pages_viewed"].astype(int) > 0)
web_temp[~check]

,session_id,customer_id,session_timestamp,device,traffic_source,pages_viewed,session_duration_sec,converted


## 6.7 session_duration_sec

In [139]:
web_temp["session_duration_sec"].value_counts(dropna=False)

,count
session_duration_sec,
NaN,2426
45.0,561
68.0,541
66.0,540
61.0,536
...,...
119922.0,1
973.0,1
94111.0,1


### Check and Finding: There are no negative session durations, and the column itself is already float. Missing values are not a significant issue and can be ignored aswell. Significant sessions can be flagged later after more careful inspection.

In [140]:
check = web_temp["session_duration_sec"].notna() & (web_temp["session_duration_sec"] > 0)
web_temp[~check & web_temp["session_duration_sec"].notna()]

,session_id,customer_id,session_timestamp,device,traffic_source,pages_viewed,session_duration_sec,converted


## 6.8 converted

### Issue 41: There are many different labels for the same label.

In [141]:
web_temp["converted"].value_counts(dropna=False)

,count
converted,
0,48218
N,4920
FALSE,4751
1,1733
Y,206
TRUE,172


### Proposed Fix 41: Relabel values to boolean True/False values.

# 7 cross-table checks

## 7.1 order items unit price should match product price in product table

In [142]:
oi_temp.head()

,order_item_id,order_id,product_id,quantity,unit_price
0,521656,ORD-210489,P236,1,35.62
1,502093,ORD-201014,P196,1,132.05
2,538709,ORD-218719,P109,1,106.65
3,549040,ORD-223642,P184,2,275.89
4,533522,ORD-216198,P257,1,65.43


In [143]:
merged_prices = oi_temp.merge(products_temp, how = "left", on = "product_id")

### Issue 42: Many unit prices are missing in order items table when they are present in products table. Also, unit prices in order items table appear to be mistyped whereas products has the correct unit price.

In [144]:
check = merged_prices["unit_price_x"].astype(float) == merged_prices["unit_price_y"].astype(float)
merged_prices[~check]

,order_item_id,order_id,product_id,quantity,unit_price_x,product_name,category,unit_price_y,cost_price,stock_qty,supplier,launch_date,is_active
50,537416,ORD-218101,P196,2,NaN,Valley 45L Trekking Pack,Backpacks,132.05,84.39,310.0,Summit Wholesale Ltd,2025-03-26,1
123,541743,ORD-220151,P246,2,NaN,Peak Sun Hat,Accessories,43.29,27.19,317.0,Northern Gear,2022-07-21,1
153,545240,ORD-221822,P203,1,1219.1,Storm Duffel Bag,backpacks,121.91,73.34,289.0,Northern Gear,20.10.24,FALSE
195,510166,ORD-204939,P158,2,NaN,Ridge 2-Person Tent,Camping,191.50,69.52,400.0,Alpine Supply Co,2024-06-07,FALSE
228,530162,ORD-214583,P172,2,NaN,Trail Sleeping Mat,Camping,103.25,65.78,70.0,Northern Gear,2024-09-29,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
51647,550446,ORD-224300,P145,1,NaN,Trail Winter Boot,Foot wear,97.59,61.90,251.0,BlueRidge Textiles Ltd,2023-11-22,Y
51650,521837,ORD-210575,P231,2,346.9,Summit Neck Gaiter,Base Layers,34.69,19.88,78.0,Northern Gear,2022-07-07,N
51723,529033,ORD-214052,P148,1,NaN,Ridge Winter Boot,Foot wear,97.70,53.60,-22.0,TREKCO LTD,2022-08-27,TRUE
51780,527240,ORD-213176,P236,2,NaN,Storm Trekking Poles,Accessories,35.62,14.90,75.0,Alpine Supply Co,11/08/2025,TRUE


### Proposed Fix 42: replace all missing/incorrect values for unit price in the order items table with the correct unit price in the products table.

### Issue 43: The old customer ids were removed from the customer table. The old ones need still exist in orders and web sessions

### Proposed Fix 43: Replace old customer ids with the most up to date customer id.

### Issue 44: There are many refund amounts that are higher than possible. Many are lower than possible but it is unclear whether this is due to full vs partial refunds.

In [145]:
ret_details = returns.merge(oi_temp, how="left", on="order_item_id")
ret_details[
        (ret_details["refund_amount"] > ret_details["quantity"] * ret_details["unit_price"]) |
        ((ret_details["unit_price"] > ret_details["refund_amount"]) )
    ]

,return_id,order_item_id,order_id_x,return_date,reason,refund_amount,resolution,order_id_y,product_id,quantity,unit_price
3,R7004,528860,ORD-213965,TBC,Too Small,586.67,refund,ORD-213965,P134,1,188.89
4,R7005,511660,ORD-205685,2026-09-30,arrived damaged,20.76,refund,ORD-205685,P229,1,41.52
5,R7006,542424,ORD-220474,2026-04-29,no longer needed,112.86,Refund,ORD-220474,P126,1,225.73
7,R7008,513659,ORD-206657,2025-01-30,TOO SMALL,16.39,Exchange,ORD-206657,P214,1,32.78
8,R7009,504446,ORD-202187,01 Jun 2026,too big,15.31,Refund,ORD-202187,P261,1,30.63
...,...,...,...,...,...,...,...,...,...,...,...
2783,R9784,540717,ORD-219659,2026-07-01,not waterproof,37.51,refund,ORD-219659,P104,1,75.01
2785,R9786,512992,ORD-206333,2025-08-24,not as described,4.90,REFUND,ORD-206333,P238,1,9.80
2791,R9792,511154,ORD-205433,2026-02-13,bought by mistake,221.31,Exchange,ORD-205433,P128,1,276.64
2793,R9794,506881,ORD-203337,"May 14, 2026",looks different to photo,46.27,Store credit,ORD-203337,P237,2,57.84


### Proposed fix: Cap the refund amount at the unit price.

###Issue 45: The cost price for 6 products are higher than what they are being sold for.

In [146]:
products_temp[products_temp["unit_price"] < products_temp["cost_price"]]

,product_id,product_name,category,unit_price,cost_price,stock_qty,supplier,launch_date,is_active
40,P141,Peak Hiking Sandal,Footwear,139.14,194.80,313,BlueRidge Textiles,2025-04-30,FALSE
81,P182,Storm Water Filter,Camping,250.06,350.08,103,BlueRidge Textiles,06-27-2024,FALSE
83,P184,Summit Water Filter,Camping,275.89,386.25,365,Alpine Supply Co,2022-09-02,0
128,P229,Peak Neck Gaiter,Base Layers,41.52,58.13,386,BlueRidge Textiles,06-13-2022,N
139,P240,Coastal Beanie,Accesories,44.87,62.82,250,BlueRidge Textiles,2024-01-05,FALSE
151,P252,Ridge Insulated Bottle,Accessories,19.56,27.38,200,summit wholesale,2024-11-15,TRUE
